# Wide Toy Semiconductor Feature Booster

이 노트북은 order별 10,000개 이상 candidate feature를 생성하고, `DefectAFeatureEvidenceBooster`로 order별 top feature를 뽑는 예제입니다.

- `sensor_*`, `measure_*`, `midproc_*`, 설비/공정시간 계열 feature가 섞여 있습니다.
- `eds_*`, `sim_*` 컬럼은 y/검증용이라 feature에서 제외합니다.
- 처음 실행이 느리면 아래 설정 셀에서 `FEATURES_PER_ORDER = 1000`으로 낮춰 먼저 확인하세요.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "examples"))

DATA_DIR = PROJECT_ROOT / "data" / "toy_semiconductor_wide"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "wide_toy_semiconductor_booster"

ORDERS = [1, 2, 3]
ROWS_PER_ORDER = 240
FEATURES_PER_ORDER = 10000
TOP_K = 10
BOOTSTRAP_ROUNDS = 5

print(PROJECT_ROOT)

## 1. Generate Wide Toy Dataset

order별 CSV를 생성합니다. 기본값은 order 3개, order별 10,000개 generated feature입니다.

In [ ]:
from generate_wide_toy_semiconductor_dataset import generate_dataset

summary = generate_dataset(
    output_dir=DATA_DIR,
    orders=max(ORDERS),
    rows=ROWS_PER_ORDER,
    features_per_order=FEATURES_PER_ORDER,
    overwrite=True,
)
summary

## 2. Run Evidence Booster

CatBoost SHAP probe는 기본적으로 끕니다. 10,000개 feature 상황에서는 먼저 evidence ranking이 잘 도는지 보는 것이 좋습니다.

In [ ]:
from run_feature_booster_on_wide_toyset import run_booster

result = run_booster(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    order_list=ORDERS,
    top_k=TOP_K,
    bootstrap_rounds=BOOTSTRAP_ROUNDS,
    catboost_enabled=False,
)

result[[
    "order_id",
    "final_rank",
    "feature_name",
    "final_score",
    "presence_type",
    "direction",
    "evidence_reason",
]]

## 3. Inspect Saved Report

모든 order 결과는 `outputs/wide_toy_semiconductor_booster/combined_feature_evidence.csv`에 저장됩니다.

In [ ]:
import pandas as pd

combined_path = OUTPUT_DIR / "combined_feature_evidence.csv"
combined = pd.read_csv(combined_path)
combined.head(20)

## 4. Quick Interpretation

Toyset에서는 A 관련 signal이 심어져 있으므로 아래 계열이 상위에 자주 보이면 정상입니다.

- `midproc_a_residue_count_like_*`
- `sensor_a_pressure_like_*`
- `sensor_a_plasma_instability_like_*`
- `measure_a_cd_*`
- `bad_only_a_sparse_signature_*`